# Solutions · 14 · Moving boundaries and transformed domains

Worked solutions to the exercises of [notebook 14](../notebooks/14_moving_boundaries_and_transformed_domains.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, grids, boundary as bc, stencils as st, stiffness
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Price the Black-Scholes call on a grid in z = log S (uniform in z from log 1 to log 400): the equation becomes constant-coefficient. Compare the error with the S-grid at equal numbers of points, especially near the strike.

In [2]:
K, r, sigma, T = 100.0, 0.05, 0.2, 1.0
from scipy.special import ndtr
def bs(S, tau):
    with np.errstate(divide="ignore"):
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return np.where(S > 0, S * ndtr(d1) - K * np.exp(-r * tau) * ndtr(d2), 0.0)
n = 400
for name, grid_kind in (("uniform in S", "S"), ("uniform in z = log S", "z")):
    if grid_kind == "S":
        p = pr.black_scholes(n=n); s = p.solve("bdf", 1e-3, order=2, n_out=1); S, V = p.x, s.y[-1]
    else:
        z = np.linspace(np.log(1.0), np.log(400.0), n + 1); dz = z[1] - z[0]; S = np.exp(z)
        def rhs(tau, V):
            Vp = np.r_[2 * V[0] - V[1], V, 2 * V[-1] - V[-2]]
            Vz = (Vp[2:] - Vp[:-2]) / (2 * dz); Vzz = (Vp[2:] - 2 * Vp[1:-1] + Vp[:-2]) / dz**2
            out = 0.5 * sigma**2 * Vzz + (r - 0.5 * sigma**2) * Vz - r * V         # constant coefficients in z
            out[0] = 0.0; out[-1] = r * K * np.exp(-r * tau)
            return out
        V = ti.bdf(rhs, np.maximum(S - K, 0.0), (0, T), 1e-3, 2, band=(1, 1), n_out=1).y[-1]
    near = (S > 80) & (S < 120)
    print(f"{name:<22}: max error {np.max(np.abs(V - bs(S, T))[(S > 1) & (S < 390)]):.2e}; near the strike (80 < S < 120) {np.max(np.abs(V - bs(S, T))[near]):.2e}; spacing at the strike {np.diff(S)[np.argmin(np.abs(S - K))]:.2f}")

uniform in S          : max error 2.50e-03; near the strike (80 < S < 120) 2.50e-03; spacing at the strike 1.00


uniform in z = log S  : max error 2.20e-03; near the strike (80 < S < 120) 2.02e-03; spacing at the strike 1.50


In z = log S the equation has constant coefficients, but a grid uniform in z from S = 1 to 400 is *coarser*
at the strike (spacing 1.5 against 1.0) because it spends half its points below S = 20, where the option is
worthless; its error near the strike is nevertheless slightly smaller, as the constant-coefficient operator
is better behaved. Neither grid is good: the error lives at the strike's kink, and a grid stretched around
S = K is what a pricing code uses.

## Exercise 2

Solve the Stefan problem with St = 0.1 and St = 10 and compare the front laws 2 lambda sqrt(t); what limits the time step in each case (look at the spectrum of the transformed system, which includes the front ODE)?

In [3]:
for St in (0.1, 10.0):
    p = pr.stefan(St=St)
    s = p.solve("dopri", rtol=1e-8, atol=1e-10, n_out=1)
    lam = stiffness.spectrum(p.rhs, p.y0); lam = lam[np.abs(lam) > 1e-9]
    print(f"St = {St:4}: lambda = {p.params['lambda']:.4f}, front at t = 1: {s.y[-1, -1]:.4f} (2 lambda sqrt(t) = {2 * p.params['lambda'] * np.sqrt(1 + p.params['t0']):.4f}); {s.steps} steps; most negative eigenvalue {lam.real.min():.0f} (diffusion on xi/s: ~ -4/(dxi s)^2)")

St =  0.1: lambda = 1.2570, front at t = 1: 2.5758 (2 lambda sqrt(t) = 2.5760); 5826 steps; most negative eigenvalue -126546 (diffusion on xi/s: ~ -4/(dxi s)^2)


St = 10.0: lambda = 0.2200, front at t = 1: 0.4509 (2 lambda sqrt(t) = 0.4509); 190159 steps; most negative eigenvalue -4130596 (diffusion on xi/s: ~ -4/(dxi s)^2)


In this formulation the Stefan number divides the front speed (ṡ = −T_x/St), so a *large* St means a slow
front: λ is small, the liquid stays thin, and the transformed diffusion term 1/s² is huge - the system is far
stiffer and the explicit solver takes thirty times more steps. A small St lets the front run, s grows, and
the effective diffusivity on the fixed ξ-grid falls with it. The front ODE itself is never the stiff part;
for slow fronts an implicit integrator on the transformed system is the right tool.

## Exercise 3

**Implement it yourself:** write the steady dispersed reactor (-c_x + c_xx/Pe - Da c = 0 with the Danckwerts Robin inlet and a zero-gradient outlet) on a grid stretched towards the inlet, solve it by a direct linear solve, and show that the integral balance holds: inlet flux (c - c_x/Pe = 1) minus outlet flux (c(1)) equals the consumption Da * integral of c.

In [4]:
Pe, Da = 10.0, 2.0
exact = pr.plug_flow_reactor(Pe=Pe, Da=Da).exact
for name, g in (("uniform, 80 intervals", grids.uniform(0, 1, 80)), ("stretched towards the inlet, 80 intervals", grids.stretched(0, 1, 80, cluster=0.2, strength=0.7))):
    D1, D2 = st.diff_operator(g.x, 1, 2).to_dense(), st.diff_operator(g.x, 2, 2).to_dense()
    A = -D1 + D2 / Pe - Da * np.eye(g.n)
    A[0, :] = -D1[0, :] / Pe; A[0, 0] += 1.0                  # inlet: c - c_x/Pe = 1 (Danckwerts)
    A[-1, :] = D1[-1, :]                                      # outlet: c_x = 0
    b = np.zeros(g.n); b[0] = 1.0
    c = np.linalg.solve(A, b)
    flux_in, flux_out = 1.0, c[-1]                            # c - c_x/Pe at the inlet is imposed; at the outlet c_x = 0 leaves c
    consumed = Da * g.integrate(c)
    print(f"{name:<42}: spacing {np.diff(g.x).min():.4f}..{np.diff(g.x).max():.4f}; max error vs Danckwerts {np.max(np.abs(c - exact(g.x))):.1e}; balance: in - out - consumed = {flux_in - flux_out - consumed:+.1e}")

uniform, 80 intervals                     : spacing 0.0125..0.0125; max error vs Danckwerts 1.4e-04; balance: in - out - consumed = -2.5e-04
stretched towards the inlet, 80 intervals : spacing 0.0042..0.0638; max error vs Danckwerts 2.6e-03; balance: in - out - consumed = -3.4e-03


The steady reactor's integral balance - what enters minus what leaves equals what reacts - holds to the
discretisation error, which is the check that the boundary rows are consistent with the interior; the
conversion matches Danckwerts' closed form. The stretched grid is *worse* than the uniform one: at Pe = 10 the
inlet jump is 1/Pe = 0.1 wide and a uniform spacing of 0.0125 resolves it already, so clustering points there
only coarsens the rest of the reactor, where the second-order stencils on a non-uniform grid lose accuracy.
Stretching pays when a layer is narrower than the uniform spacing (notebook 14's boundary layer at Pe = 200),
not before.